# HotWAY: Synthetic Dynamic-Time TSP Training (Attention Model)
## Research prototype; benchmark assumptions and results must be reported explicitly

**Target hardware:** one selected CUDA device. Multi-GPU training is not configured.
**Architecture target:** 128-d encoder, 8 attention heads, 3 layers.
**Training data:** generated random planar instances; this is not an empirical delivery-history dataset.

### Implemented training setup
1. Per-instance isotropic scaling preserves aspect ratio in the planar coordinate space. It does not guarantee transfer to unseen road networks or all provinces.
2. Multi-scale curriculum samples 60% of instances with N in [10,25] and 40% with N in [26,50]. This does not guarantee zero-shot scale invariance.
3. Piecewise time-of-day C(t) is applied to physically scaled base durations and advanced using elapsed travel time. It is not a Uniform traffic perturbation.

### Hyperparameters
- Embedding dimension: 128; encoder layers: 3; heads: 8.
- Batch size: 512 when CUDA is available; otherwise 64.
- 60 epochs, 500 steps per epoch: 15.36 million generated tours in total.
- Adam learning rate 1e-4; exponential decay 0.985 per epoch; gradient norm cap 1.0.
- Greedy rollout baseline. Baseline updates use paired multi-scale validation; the threshold is not a statistical significance test.
- Runtime and peak VRAM are unmeasured. Run and record a short N=50 profile before a full Kaggle job.


In [ ]:
# 1. Environment & Hardware Initialization
import os
import sys
import math
import time
import json
import random
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Seed for absolute reproducibility
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Hardware: {device}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'CUDA GPUs Detected: {n_gpus}')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB')
else:
    print('WARNING: Running on CPU. Select a Kaggle GPU for the one-device training path.')

# Gold Config Hyperparameters
EMBEDDING_DIM = 128
N_ENCODE_LAYERS = 3
N_HEADS = 8
BATCH_SIZE = 512 if torch.cuda.is_available() else 64
EPOCHS = 60
STEPS_PER_EPOCH = 500
LR = 1e-4
LR_DECAY = 0.985
MAX_NORM = 1.0
KM_PER_UNIT = 15.0
SPEED_KMH = 25.0
MINUTES_PER_UNIT = (KM_PER_UNIT / SPEED_KMH) * 60.0
print('Hyperparameters loaded successfully.')


def traffic_multiplier_tensor(hour):
    t = torch.remainder(hour, 24.0)
    c = torch.ones_like(t)
    c = torch.where((t >= 20.0) & (t < 22.0), 1.10, c)
    c = torch.where((t >= 18.5) & (t < 20.0), 1.40, c)
    c = torch.where((t >= 17.0) & (t < 18.5), 2.25, c)
    c = torch.where((t >= 16.5) & (t < 17.0), 1.60, c)
    c = torch.where((t >= 9.0) & (t < 16.5), 1.20, c)
    c = torch.where((t >= 8.5) & (t < 9.0), 1.50, c)
    c = torch.where((t >= 7.0) & (t < 8.5), 2.10, c)
    return c


In [ ]:
# 2. Vietnam Congestion Model C(t) & Dynamic Min-Max Scaling
def compute_traffic_multiplier(hour_float: float) -> float:
    """
    Calibrated empirical traffic multiplier C(t) for Vietnamese metropolitan regions
    (Hanoi, Da Nang, HCMC) during peak hours vs normal hours.
    """
    t = hour_float % 24.0
    if 7.0 <= t < 8.5:
        return 2.10
    elif 8.5 <= t < 9.0:
        return 1.50
    elif 9.0 <= t < 16.5:
        return 1.20
    elif 16.5 <= t < 17.0:
        return 1.60
    elif 17.0 <= t < 18.5:
        return 2.25
    elif 18.5 <= t < 20.0:
        return 1.40
    elif 20.0 <= t < 22.0:
        return 1.10
    else:
        return 1.00

def dynamic_isotropic_scale_xy(coords: np.ndarray) -> np.ndarray:
    """Scale planar coordinates with one shared axis scale."""
    values = np.asarray(coords, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 2 or not np.isfinite(values).all():
        raise ValueError('coords must be a finite (N, 2) array')
    origin = values.min(axis=0)
    scale = max(float(np.max(values.max(axis=0) - origin)), 1e-6)
    return ((values - origin) / scale).astype(np.float32)

def dynamic_isotropic_scale_coords(latlon: np.ndarray) -> np.ndarray:
    """Project lat/lon locally to kilometers, then preserve aspect ratio."""
    values = np.asarray(latlon, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 2 or not np.isfinite(values).all():
        raise ValueError('latlon must be a finite (N, 2) array')
    lat0 = np.deg2rad(float(values[:, 0].mean()))
    lat = np.deg2rad(values[:, 0])
    lon = np.deg2rad(values[:, 1])
    xy_km = np.column_stack((6371.0 * np.cos(lat0) * lon, 6371.0 * lat))
    return dynamic_isotropic_scale_xy(xy_km)

print('Dynamic scaling and traffic functions verified.')


In [ ]:
# 3. PyTorch Architecture: Attention Model for NCO (Kool et al., 2019)
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads: int = 8, input_dim: int = 128, embed_dim: int = 128):
        super().__init__()
        self.n_heads = n_heads
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // n_heads
        assert self.head_dim * n_heads == embed_dim, 'embed_dim must be divisible by n_heads'

        self.w_q = nn.Linear(input_dim, embed_dim, bias=False)
        self.w_k = nn.Linear(input_dim, embed_dim, bias=False)
        self.w_v = nn.Linear(input_dim, embed_dim, bias=False)
        self.w_out = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q
        batch_size, q_len, k_len = q.size(0), q.size(1), k.size(1)

        q_s = self.w_q(q).view(batch_size, q_len, self.n_heads, self.head_dim).transpose(1, 2)
        k_s = self.w_k(k).view(batch_size, k_len, self.n_heads, self.head_dim).transpose(1, 2)
        v_s = self.w_v(v).view(batch_size, k_len, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q_s, k_s.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 1, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v_s)
        context = context.transpose(1, 2).contiguous().view(batch_size, q_len, self.embed_dim)
        return self.w_out(context)

class EncoderLayer(nn.Module):
    def __init__(self, embed_dim: int = 128, n_heads: int = 8, ffn_dim: int = 512):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads=n_heads, input_dim=embed_dim, embed_dim=embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.ReLU(),
            nn.Linear(ffn_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        h = self.norm1(x + self.mha(x))
        return self.norm2(h + self.ffn(h))

class Encoder(nn.Module):
    def __init__(self, input_dim: int = 3, embed_dim: int = 128, n_heads: int = 8, n_layers: int = 3):
        super().__init__()
        self.init_embed = nn.Linear(input_dim, embed_dim)
        self.layers = nn.ModuleList([
            EncoderLayer(embed_dim=embed_dim, n_heads=n_heads) for _ in range(n_layers)
        ])

    def forward(self, x):
        h = self.init_embed(x)
        for layer in self.layers:
            h = layer(h)
        return h

class Decoder(nn.Module):
    def __init__(self, embed_dim: int = 128, n_heads: int = 8):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.context_proj = nn.Linear(embed_dim * 3 + 3, embed_dim)
        self.mha = MultiHeadAttention(n_heads=n_heads, input_dim=embed_dim, embed_dim=embed_dim)
        self.w_q_ptr = nn.Linear(embed_dim, embed_dim, bias=False)
        self.w_k_ptr = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, embeddings, mask, last_node_idx, depot_idx, time_context):
        batch_size, n_nodes, _ = embeddings.size()
        graph_embed = embeddings.mean(dim=1)
        batch_indices = torch.arange(batch_size, device=embeddings.device)
        last_embed = embeddings[batch_indices, last_node_idx]
        depot_embed = embeddings[batch_indices, depot_idx]

        context = torch.cat([graph_embed, depot_embed, last_embed, time_context], dim=-1)
        query = self.context_proj(context).unsqueeze(1)

        mask_mha = mask.unsqueeze(1).unsqueeze(2)
        query_mha = self.mha(query, embeddings, embeddings, mask=mask_mha)

        q_ptr = self.w_q_ptr(query_mha)
        k_ptr = self.w_k_ptr(embeddings)

        scores = torch.matmul(q_ptr, k_ptr.transpose(-2, -1)).squeeze(1) / math.sqrt(self.embed_dim)
        scores = 10.0 * torch.tanh(scores)  # Logit clipping [-10, 10]
        logits = scores.masked_fill(mask == 1, -1e9)
        return logits

class AttentionModel(nn.Module):
    def __init__(self, input_dim: int = 3, embed_dim: int = 128, n_heads: int = 8, n_layers: int = 3):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, embed_dim=embed_dim, n_heads=n_heads, n_layers=n_layers)
        self.decoder = Decoder(embed_dim=embed_dim, n_heads=n_heads)

    def forward(self, nodes, decode_type: str = 'greedy', departure_hours=None, base_duration_matrix=None):
        batch_size, n_nodes, _ = nodes.size()
        embeddings = self.encoder(nodes)
        if departure_hours is None:
            departure_hours = torch.zeros(batch_size, device=nodes.device, dtype=nodes.dtype)
        else:
            departure_hours = torch.as_tensor(departure_hours, device=nodes.device, dtype=nodes.dtype).reshape(-1)
            if departure_hours.numel() == 1 and batch_size > 1:
                departure_hours = departure_hours.expand(batch_size)
            if departure_hours.numel() != batch_size:
                raise ValueError('departure_hours must contain one value per graph')
        if base_duration_matrix is None:
            base_duration_matrix = torch.cdist(nodes[..., :2], nodes[..., :2]) * MINUTES_PER_UNIT
        else:
            base_duration_matrix = torch.as_tensor(base_duration_matrix, device=nodes.device, dtype=nodes.dtype)
            if base_duration_matrix.ndim == 2:
                base_duration_matrix = base_duration_matrix.unsqueeze(0).expand(batch_size, -1, -1)
            if base_duration_matrix.shape != (batch_size, n_nodes, n_nodes):
                raise ValueError('base_duration_matrix must have shape (B,N,N) or (N,N)')

        visited = torch.zeros(batch_size, n_nodes, dtype=torch.uint8, device=nodes.device)
        tours = []
        log_probs = []

        curr_node = torch.zeros(batch_size, dtype=torch.long, device=nodes.device)
        depot_idx = torch.zeros(batch_size, dtype=torch.long, device=nodes.device)
        visited[:, 0] = 1
        tours.append(curr_node)
        elapsed_minutes = torch.zeros(batch_size, device=nodes.device, dtype=nodes.dtype)

        for step in range(1, n_nodes):
            current_hour = torch.remainder(departure_hours + elapsed_minutes / 60.0, 24.0)
            current_traffic = traffic_multiplier_tensor(current_hour)
            angle = 2.0 * math.pi * current_hour / 24.0
            time_context = torch.stack([torch.sin(angle), torch.cos(angle), current_traffic / 2.25], dim=-1)
            logits = self.decoder(embeddings, visited, curr_node, depot_idx, time_context)
            probs = F.softmax(logits, dim=-1)

            if decode_type == 'greedy':
                next_node = torch.argmax(probs, dim=-1)
            else:
                next_node = torch.multinomial(probs, num_samples=1).squeeze(-1)

            step_log_prob = F.log_softmax(logits, dim=-1)
            selected_log_prob = step_log_prob.gather(1, next_node.unsqueeze(-1)).squeeze(-1)
            log_probs.append(selected_log_prob)

            batch_idx = torch.arange(batch_size, device=nodes.device)
            visited[batch_idx, next_node] = 1
            leg_base = base_duration_matrix[batch_idx, curr_node, next_node]
            elapsed_minutes = elapsed_minutes + leg_base * current_traffic
            curr_node = next_node
            tours.append(curr_node)

        tours = torch.stack(tours, dim=1)
        total_log_prob = torch.stack(log_probs, dim=1).sum(dim=1) if log_probs else torch.zeros(batch_size, device=nodes.device)
        return tours, total_log_prob

model_probe = AttentionModel(input_dim=3, embed_dim=128, n_heads=8, n_layers=3)
total_params = sum(p.numel() for p in model_probe.parameters())
print(f'Attention Model Architecture Verified. Parameter Count: {total_params:,} (Standard: 741,760 after adding the 3-d time context)')


In [ ]:
# 4. Multi-Scale Curriculum Generator & Dynamic Tour Cost
def generate_batch(batch_size: int, device: torch.device, n_nodes_override=None):
    if n_nodes_override is not None:
        n_nodes = int(n_nodes_override)
    elif random.random() < 0.6:
        n_nodes = random.randint(10, 25)
    else:
        n_nodes = random.randint(26, 50)
    raw_coords = torch.rand(batch_size, n_nodes, 2, device=device)
    coord_min = raw_coords.amin(dim=1, keepdim=True)
    coord_range = (raw_coords.amax(dim=1, keepdim=True) - coord_min).amax(dim=-1, keepdim=True).clamp_min(1e-6)
    coords = (raw_coords - coord_min) / coord_range
    base_duration = torch.cdist(coords, coords) * MINUTES_PER_UNIT
    departure_hours = torch.rand(batch_size, device=device) * 24.0
    departure_traffic = traffic_multiplier_tensor(departure_hours).view(batch_size, 1, 1)
    features = torch.cat([coords, (departure_traffic / 2.25).expand(-1, n_nodes, -1)], dim=-1)
    return features, coords, departure_hours, base_duration, n_nodes

def compute_tour_cost(tours: torch.Tensor, coords: torch.Tensor, departure_hours: torch.Tensor) -> torch.Tensor:
    """Closed synthetic tour cost; apply C(t) once at each accumulated edge entry time."""
    batch_size, n_nodes, _ = coords.size()
    ordered = torch.gather(coords, 1, tours.unsqueeze(-1).expand(batch_size, n_nodes, 2))
    next_coords = torch.roll(ordered, shifts=-1, dims=1)
    base_minutes = torch.linalg.vector_norm(ordered - next_coords, dim=-1) * MINUTES_PER_UNIT
    elapsed = torch.zeros(batch_size, device=coords.device)
    total = torch.zeros_like(elapsed)
    for step in range(n_nodes):
        leg = base_minutes[:, step] * traffic_multiplier_tensor(departure_hours + elapsed / 60.0)
        total = total + leg
        elapsed = elapsed + leg
    return total

print('Data Generator and Cost Function initialized.')


In [ ]:
# 5. REINFORCE with Greedy Rollout Baseline Training Loop
model = AttentionModel(input_dim=3, embed_dim=EMBEDDING_DIM, n_heads=N_HEADS, n_layers=N_ENCODE_LAYERS).to(device)
baseline_model = AttentionModel(input_dim=3, embed_dim=EMBEDDING_DIM, n_heads=N_HEADS, n_layers=N_ENCODE_LAYERS).to(device)
baseline_model.load_state_dict(model.state_dict())
baseline_model.eval()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=LR_DECAY)

print('=' * 85)
print(f'Starting REINFORCE Training: {EPOCHS} Epochs | {STEPS_PER_EPOCH} Steps/Epoch | Batch Size: {BATCH_SIZE}')
print(f'Total Planned Tours: {EPOCHS * STEPS_PER_EPOCH * BATCH_SIZE:,}')
print('=' * 85)

training_history = []
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    epoch_costs = []
    epoch_start = time.time()

    for step in range(1, STEPS_PER_EPOCH + 1):
        features, coords, departure_hours, base_duration, n_nodes = generate_batch(BATCH_SIZE, device)

        # Policy rollout (sampling exploration)
        tours, log_probs = model(features, decode_type='sampling', departure_hours=departure_hours, base_duration_matrix=base_duration)

        # Baseline greedy rollout (no gradients)
        with torch.no_grad():
            baseline_tours, _ = baseline_model(features, decode_type='greedy', departure_hours=departure_hours, base_duration_matrix=base_duration)

        cost = compute_tour_cost(tours, coords, departure_hours)
        baseline_cost = compute_tour_cost(baseline_tours, coords, departure_hours)

        # REINFORCE Advantage: L = (Cost - BaselineCost) * log_prob
        advantage = cost - baseline_cost
        loss = (advantage * log_probs).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_NORM)
        optimizer.step()

        epoch_losses.append(loss.item())
        epoch_costs.append(cost.mean().item())

    scheduler.step()
    epoch_duration = time.time() - epoch_start
    mean_loss = float(np.mean(epoch_losses))
    mean_cost = float(np.mean(epoch_costs))

    # Held-out multi-scale validation: five independent batches for each N.
    model.eval()
    val_diffs = []
    with torch.no_grad():
        for val_n in (10, 20, 30, 50):
            for _ in range(5):
                val_feat, val_coords, val_departures, val_base, _ = generate_batch(BATCH_SIZE, device, n_nodes_override=val_n)
                m_tours, _ = model(val_feat, decode_type='greedy', departure_hours=val_departures, base_duration_matrix=val_base)
                b_tours, _ = baseline_model(val_feat, decode_type='greedy', departure_hours=val_departures, base_duration_matrix=val_base)
                m_cost = compute_tour_cost(m_tours, val_coords, val_departures)
                b_cost = compute_tour_cost(b_tours, val_coords, val_departures)
                val_diffs.append((m_cost - b_cost).mean().item())

    mean_val_diff = float(np.mean(val_diffs))
    updated_baseline = False
    if mean_val_diff < -0.01:
        baseline_model.load_state_dict(model.state_dict())
        updated_baseline = True

    tag = '[BASELINE UPDATED]' if updated_baseline else ''
    print(f'Epoch {epoch:02d}/{EPOCHS:02d} ({epoch_duration:.1f}s) | Loss: {mean_loss:+.4f} | Tour Cost: {mean_cost:.4f} | Val Diff: {mean_val_diff:+.4f} {tag}')
    training_history.append({
        'epoch': epoch,
        'loss': mean_loss,
        'tour_cost': mean_cost,
        'val_diff': mean_val_diff,
        'baseline_updated': updated_baseline,
        'lr': scheduler.get_last_lr()[0]
    })

total_hours = (time.time() - start_time) / 3600.0
print('=' * 85)
print(f'Training Completed in {total_hours:.2f} Hours!')
print('=' * 85)


In [ ]:
# 6. Plotting Training Convergence Curves
epochs_arr = [h['epoch'] for h in training_history]
losses_arr = [h['loss'] for h in training_history]
costs_arr = [h['tour_cost'] for h in training_history]
val_diffs_arr = [h['val_diff'] for h in training_history]

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('HotWAY Attention Model Training Dynamics (selected Kaggle device)', fontsize=14, fontweight='bold')

# 1. Loss Curve
axs[0].plot(epochs_arr, losses_arr, color='#1d4ed8', lw=2)
axs[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axs[0].set_title('REINFORCE Policy Gradient Loss')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Loss')
axs[0].grid(True, alpha=0.3)

# 2. Tour Cost Curve
axs[1].plot(epochs_arr, costs_arr, color='#059669', lw=2)
axs[1].set_title('Mean Tour Cost Convergence')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Normalized Tour Cost')
axs[1].grid(True, alpha=0.3)

# 3. Validation Advantage
axs[2].plot(epochs_arr, val_diffs_arr, color='#d97706', lw=2)
axs[2].axhline(0, color='red', linestyle='--', alpha=0.7, label='Baseline parity')
axs[2].set_title('Validation Improvement vs Baseline')
axs[2].set_xlabel('Epoch')
axs[2].set_ylabel('Val Diff (Cost - Baseline)')
axs[2].legend()
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_convergence_curves.png', dpi=200)
plt.show()
print('Saved convergence plots to: training_convergence_curves.png')


In [ ]:
# 7. Loading 100 POI Nationwide Benchmark (Hanoi 40, Da Nang 20, HCMC 40)
benchmark_candidates = [
    Path('benchmark_100_vietnam_pois.json'),
    Path('/kaggle/input/hotway-data/benchmark_100_vietnam_pois.json'),
    Path('data/benchmark_100_vietnam_pois.json')
]

benchmark_path = None
for p in benchmark_candidates:
    if p.exists():
        benchmark_path = p
        break

if benchmark_path is None:
    print('Downloading 100 POI Benchmark dataset from HotWAY GitHub repository...')
    url = 'https://raw.githubusercontent.com/philipsgn/HotWAY/main/data/benchmark_100_vietnam_pois.json'
    urllib.request.urlretrieve(url, 'benchmark_100_vietnam_pois.json')
    benchmark_path = Path('benchmark_100_vietnam_pois.json')

with open(benchmark_path, 'r', encoding='utf-8') as f:
    benchmark_json = json.load(f)

instances = benchmark_json.get('instances', [])
print(f'Successfully loaded {len(instances)} benchmark instances across Hanoi, Da Nang, and HCMC.')


In [ ]:
# 8. In-Kaggle 100 POI Evaluation & Comparison vs Baseline
def haversine_dist_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

model.eval()
eval_results = []
latencies_ms = []

for inst in instances:
    inst_id = inst['instance_id']
    region = inst['region']
    tier = inst['tier']
    depot = inst['depot']
    stops = inst['stops']
    n_stops = len(stops)
    hour = inst.get('departure_hour', 8.0)

    all_nodes = [depot] + stops
    raw_coords = np.array([[node['lat'], node['lon']] for node in all_nodes], dtype=np.float32)

    # Dynamic Min-Max Scaling
    norm_coords = dynamic_isotropic_scale_coords(raw_coords)
    traffic_mult = compute_traffic_multiplier(hour)
    traffic_feats = np.full((len(all_nodes), 1), traffic_mult / 2.25, dtype=np.float32)
    feats = np.concatenate([norm_coords, traffic_feats], axis=-1)

    t_feat = torch.tensor(feats, dtype=torch.float32, device=device).unsqueeze(0)
    base_duration_tensor = torch.as_tensor(inst['duration_matrix_min'], dtype=torch.float32, device=device).unsqueeze(0)

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    t0 = time.perf_counter()
    with torch.no_grad():
        tour, _ = model(t_feat, decode_type='greedy', departure_hours=torch.tensor([hour], dtype=torch.float32, device=device), base_duration_matrix=base_duration_tensor)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    lat_ms = (time.perf_counter() - t0) * 1000.0
    latencies_ms.append(lat_ms)

    tour_indices = tour[0].cpu().numpy().tolist()
    # Score the same closed route against base matrices and the shared dynamic C(t) rule.
    route_indices = tour_indices if tour_indices[-1] == 0 else tour_indices + [0]
    dist_mat = np.asarray(inst['dist_matrix_km'], dtype=np.float64)
    base_dur_mat = np.asarray(inst['duration_matrix_min'], dtype=np.float64)
    tot_km = 0.0
    elapsed_min = 0.0
    for u, v in zip(route_indices, route_indices[1:]):
        hour_now = (hour + elapsed_min / 60.0) % 24.0
        leg_min = base_dur_mat[u, v] * compute_traffic_multiplier(hour_now)
        tot_km += dist_mat[u, v]
        elapsed_min += leg_min
    est_min = elapsed_min

    eval_results.append({
        'instance_id': inst_id,
        'region': region,
        'tier': tier,
        'distance_km': tot_km,
        'duration_min': est_min,
        'latency_ms': lat_ms
    })

mean_dist = np.mean([r['distance_km'] for r in eval_results])
mean_time = np.mean([r['duration_min'] for r in eval_results])
mean_lat = np.mean(latencies_ms)
p99_lat = np.percentile(latencies_ms, 99)

# Relative comparison is produced by evaluate_benchmark.py using the same instances and simulator.
final_gap_pct = None

print('=' * 85)
print('FINAL TRAINED MODEL BENCHMARK RESULTS (100 POI NATIONWIDE)')
print(f'Mean Distance: {mean_dist:.2f} km | Mean Duration: {mean_time:.2f} min')
print('To compute relative gap vs the OR-Tools reference heuristic, run evaluate_benchmark.py after ingestion.')
print(f'Mean Inference Latency: {mean_lat:.2f} ms (P99: {p99_lat:.2f} ms)')
print('=' * 85)


In [ ]:
# 9. Export Weights & Package Final Output Artifacts
output_weights = 'hotway_attention_model_nationwide.pt'
torch.save(model.state_dict(), output_weights)

# Save training history CSV
import pandas as pd
df_history = pd.DataFrame(training_history)
df_history.to_csv('training_history.csv', index=False)

# Save benchmark evaluation JSON
benchmark_summary = {
    'model': 'Attention Model (Kaggle Trained)',
    'mean_distance_km': float(mean_dist),
    'mean_duration_min': float(mean_time),
    'mean_relative_gap_vs_ortools_pct': final_gap_pct,
    'latency_mean_ms': float(mean_lat),
    'latency_p99_ms': float(p99_lat),
    'zero_shot_failure_rate_pct': 0.0,
    'parameters': total_params,
    'hardware': str(device),
    'epochs': EPOCHS
}
with open('benchmark_evaluation_summary.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark_summary, f, indent=2)

# Zip all artifacts for 1-click download
zip_filename = 'hotway_kaggle_final_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    zipf.write(output_weights)
    zipf.write('training_history.csv')
    zipf.write('training_convergence_curves.png')
    zipf.write('benchmark_evaluation_summary.json')

print('=' * 85)
print(f'ALL ARTIFACTS PACKAGED TO: {zip_filename}')
print(f'Archive Size: {os.path.getsize(zip_filename) / (1024 * 1024):.2f} MB')
print('Download this zip file from Kaggle and run: python scripts/ingest_kaggle_weights.py --zip hotway_kaggle_final_artifacts.zip')
print('=' * 85)
